# TRASER on the paper's test sets: regenerating Table 2

**Goal:** run the released TRASER checkpoint on the three test sets of Table 2 and score it the way the paper does
(lenient semantic criterion, temporal IoU ≥ 0.5), for the **triplet, relation and object** columns.

| test set | videos | what the model gets | where it comes from |
|---|---|---|---|
| **SVG2test** | 100 (67 VIPSeg + 33 SA-V) | human masks from the SVG2 release | Hugging Face `UWGZQ/Synthetic_Visual_Genome2`; VIPSeg frames (Google Drive); SA-V mp4s (Meta's download links) |
| **PVSG** val | 62 | human masks (PNG per frame) | Hugging Face `Jingkang/PVSG` |
| **VidOR** val | 835 | human **boxes**, turned into masks with SAM 2 (as in the paper) | Hugging Face `shangxd/vidor` |

**How it runs:** one dataset per session. A background worker per GPU loads TRASER once and goes through its videos;
each finished prediction is written straight into the repo (`results/traser_bench/<dataset>/preds/`), so after you
**push**, nothing is lost when the session ends. Run the notebook again in the next session: finished videos are skipped.

**Rough time on 2 × T4:** SVG2test ~2 h, PVSG ~3 h (long videos), VidOR ~10 h (2 sessions). The evaluation (judge = Kimi K3 on NVIDIA NIM) takes a few minutes.

**Order:** SVG2test first (it tells us whether our scoring matches the paper), then PVSG, then VidOR.

In [ ]:
import os, sys, json, subprocess, time, getpass, shutil, glob
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
r = subprocess.run("python3 -m pip install -q openai huggingface_hub gdown", shell=True, capture_output=True, text=True)
print(r.stdout[-200:], r.stderr[-200:])
sys.path.insert(0, os.path.join(REPO, "tools"))
import bench_data as B, bench_eval as E, pvsg_data
BIG = "/kaggle/tmp" if os.path.isdir("/kaggle/tmp") else "/tmp"      # big downloads: not in the repo
BENCH = os.path.join(BIG, "traser_bench")                              # prepared videos (masks + labels)
RUNS = "/kaggle/working/traser_bench_runs"                             # worker logs and progress
for d in (BENCH, RUNS):
    os.makedirs(d, exist_ok=True)
print("big data →", BIG, "| free disk:", shutil.disk_usage(BIG).free // 2**30, "GB")

## 1. Keys

The judge (Kimi K3 on NVIDIA NIM) needs your **NVIDIA key**. It is typed into a hidden box once and stored in `/root/.svg2_keys`, **outside the repo**. Never type a key into a cell.

In [ ]:
KEYS = "/root/.svg2_keys"
if not (os.path.exists(KEYS) and "NVIDIA_API_KEY=" in open(KEYS).read()):
    key = getpass.getpass("NVIDIA_API_KEY (hidden): ")
    with open(KEYS, "a") as f:
        f.write(f"NVIDIA_API_KEY={key}\n")
    os.chmod(KEYS, 0o600); del key
print("NIM key stored in", KEYS)

## 2. Which test set this session

`DATASET = "svg2test"`, `"pvsg"` or `"vidor"`. `N_VIDEOS` caps how many **new** videos this session runs (`None` = all that are left).

In [ ]:
DATASET = "svg2test"
N_VIDEOS = None
done = {p.split("/")[-1][:-5] for p in glob.glob(f"results/traser_bench/{DATASET}/preds/*.json") if not p.endswith(".stats.json")}
print(f"{DATASET}: {len(done)} videos already predicted in earlier sessions (pushed), they are skipped")

## 3. Download and prepare

Run the cell for your `DATASET` only (the others do nothing).

**SVG2test** needs two kinds of videos that are not on Hugging Face:
- **VIPSeg** (67 videos): one zip from the VIPSeg authors' Google Drive, downloaded with `gdown`. If Google says the file has too many downloads, download it yourself (https://github.com/VIPSeg-Dataset/VIPSeg-Dataset), add it to the notebook as a Kaggle dataset, and set `VIPSEG_ZIP` to its path.
- **SA-V** (33 videos): Meta gives the download links after a form (https://ai.meta.com/datasets/segment-anything-video-downloads/). Download the links text file they give you, add it to the notebook (Kaggle: *Add input → upload*), and set `SAV_LINKS` to its path. Only the needed videos are kept; the shards are streamed, not stored. Leave `SAV_LINKS = None` to run the 67 VIPSeg videos first.

In [ ]:
if DATASET == "svg2test":
    VIPSEG_ZIP = None       # e.g. "/kaggle/input/vipseg/VIPSeg.zip"; None = download from Google Drive
    SAV_LINKS = None        # e.g. "/kaggle/input/sav-links/sav_train_links.txt"; None = skip the 33 SA-V videos for now
    t0 = time.time()
    SVG2_ROOT = os.path.join(BIG, "svg2")
    B.download_svg2test(SVG2_ROOT)
    rows, masks = B.svg2test_rows(SVG2_ROOT), B.svg2test_masks(SVG2_ROOT)
    vip = sorted(v for v, r in rows.items() if r["split"] == "vipseg")
    sav = sorted(v for v, r in rows.items() if r["split"] == "sav")
    print(f"SVG2test: {len(vip)} VIPSeg + {len(sav)} SA-V videos, masks for {len(masks)}  ({time.time() - t0:.0f} s)")

    videos = {}
    # --- VIPSeg: frames -> 6 fps, 720p mp4
    if VIPSEG_ZIP is None:
        VIPSEG_ZIP = os.path.join(BIG, "VIPSeg.zip")
        if not os.path.exists(VIPSEG_ZIP):
            import gdown
            gdown.download(id="1B13QUiE82xf7N6nVHclb4ErN-Zuai-sZ", output=VIPSEG_ZIP, quiet=False)
    print("VIPSeg zip:", os.path.getsize(VIPSEG_ZIP) // 2**30, "GB")
    found = B.extract_vipseg_frames(VIPSEG_ZIP, vip, os.path.join(BIG, "vipseg_frames"))
    print(f"VIPSeg frames found for {len(found)} of {len(vip)} videos")
    for v in sorted(found):
        videos[v] = B.vipseg_to_mp4(os.path.join(BIG, "vipseg_frames", v), os.path.join(BIG, "svg2test_videos", f"{v}.mp4"))
    # --- SA-V: stream the shards, keep the needed mp4s
    if SAV_LINKS:
        missing = B.fetch_sav_videos(B.sav_links(SAV_LINKS), sav, os.path.join(BIG, "svg2test_videos"))
        print("SA-V videos not found:", sorted(missing) or "none")
        for v in sav:
            p = os.path.join(BIG, "svg2test_videos", f"{v}.mp4")
            if os.path.exists(p):
                videos[v] = p

    # --- check frame counts (masks are per video frame) and prepare
    bad = []
    for v, p in sorted(videos.items()):
        _, n, _ = B.traser_frames(p)
        if n != len(masks[v]):
            bad.append((v, n, len(masks[v])))
        B.prepare_svg2test(v, rows[v], masks[v], p, BENCH, E.RESULTS)
    print(f"prepared {len(videos)} of {len(rows)} SVG2test videos")
    print("frame count differs from the mask frames (video, video frames, mask frames):", bad or "none")
    ALL_VIDEOS = sorted(videos)

In [ ]:
if DATASET == "pvsg":
    t0 = time.time()
    PVSG_ROOT = os.path.join(BIG, "pvsg")
    paths = pvsg_data.download(PVSG_ROOT, sources=("vidor", "epic_kitchen", "ego4d"))
    for name, p in paths.items():
        print(f"{name:28} {os.path.getsize(p) / 2**30:6.2f} GB")
    shutil.copy(paths["pvsg.json"], os.path.join(PVSG_ROOT, "pvsg.json"))
    anno = pvsg_data.load_anno(PVSG_ROOT)
    ALL_VIDEOS = sorted(B.pvsg_val_ids(anno))
    todo = [v for v in ALL_VIDEOS if v not in done]
    zips = {os.path.basename(p): p for p in paths.values()}
    for v, rep in pvsg_data.extract_videos(PVSG_ROOT, zips, anno, todo).items():
        chk = pvsg_data.check_video(PVSG_ROOT, anno, v)
        print(v, rep["source"], "| frames ok" if chk["ok"] else f"| !! {chk}")
    print(f"PVSG val: {len(ALL_VIDEOS)} videos ({time.time() - t0:.0f} s); masks are made by the workers")

In [ ]:
if DATASET == "vidor":
    t0 = time.time()
    VIDOR_ROOT = os.path.join(BIG, "vidor")
    B.download_vidor(VIDOR_ROOT)
    ALL_VIDEOS = sorted(B.vidor_index({"videos": os.path.join(VIDOR_ROOT, "videos"),
                                       "anno": os.path.join(VIDOR_ROOT, "anno")}))
    print(f"VidOR val: {len(ALL_VIDEOS)} videos with annotation + mp4 ({time.time() - t0:.0f} s); "
          "SAM 2 turns the boxes into masks inside the workers")

## 4. Start the workers (background, one per GPU)

Each GPU gets every second video. Logs: `traser_bench_runs/<dataset>_gpu0.log`. Running this cell again does **not** start duplicates while workers are alive. If a GPU has less than 12 GB free, shut down other notebook kernels first.

In [ ]:
running = subprocess.run("pgrep -f [t]ools/bench_run.py", shell=True, capture_output=True, text=True).stdout.split()
free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True).stdout.split()
todo = [v for v in ALL_VIDEOS if v not in done][:N_VIDEOS]
print(f"free MiB per GPU: {free} | {len(todo)} videos to run")
QUEUES = {}
if running:
    print("workers already running:", running)
elif todo:
    if any(int(f) < 12000 for f in free):
        print("!! a GPU has < 12 GB free: shut down other notebook kernels first")
    n_gpu = max(1, len(free))
    extra = {"pvsg": f"--pvsg_root {os.path.join(BIG, 'pvsg')}", "vidor": f"--vidor_root {os.path.join(BIG, 'vidor')}"}.get(DATASET, "")
    for g in range(n_gpu):
        queue = todo[g::n_gpu]
        if not queue:
            continue
        QUEUES[g] = queue
        qfile = os.path.join(RUNS, f"{DATASET}_gpu{g}.queue")
        open(qfile, "w").write("\n".join(queue))
        cmd = (f"CUDA_VISIBLE_DEVICES={g} HF_HOME=/tmp/hf_cache nohup python3 tools/bench_run.py --dataset {DATASET} "
               f"--bench {BENCH} --runs {RUNS} --gpu_name gpu{g} {extra} --videos $(cat {qfile}) "
               f">> {RUNS}/{DATASET}_gpu{g}.log 2>&1 &")
        subprocess.run(cmd, shell=True)
        print(f"GPU {g}: {len(queue)} videos")

## 5. Progress (updates by itself)

Refreshes every 30 s until both workers are finished, then scores everything automatically (section 6). Stopping this cell does **not** stop the workers; just run it again.

In [ ]:
from IPython.display import clear_output, Markdown, display

def bar(done_, total, width=30):
    total = max(int(total or 0), 1); done_ = min(int(done_ or 0), total)
    return "█" * round(width * done_ / total) + "░" * (width - round(width * done_ / total)) + f" {done_}/{total}"

def progress():
    files = sorted(glob.glob(os.path.join(RUNS, f"{DATASET}_gpu*.json")))
    if not files:
        print("no workers were started this session (nothing left to run?)")
        return True
    n_pred = len([p for p in glob.glob(f"results/traser_bench/{DATASET}/preds/*.json") if not p.endswith(".stats.json")])
    print(f"{time.strftime('%H:%M:%S')}   {DATASET}: {n_pred} of {len(ALL_VIDEOS)} videos predicted so far (all sessions)\n")
    states = []
    for p in files:
        st = json.load(open(p)); states.append(st["state"])
        print(f"{os.path.basename(p)[:-5]:16} {bar(st['done'], st['total'])}   {st['state']}   "
              f"now: {st.get('current') or '-'}   failed: {len(st['failed'])}   (last update {st.get('updated', '-')})")
        log = p[:-5] + ".log"
        if st["failed"] and os.path.exists(log):
            print("   last error:", [l for l in open(log).read().splitlines() if "FAILED" in l][-1][:200])
    return all(s == "finished" for s in states)

while True:
    clear_output(wait=True)
    if progress():
        print("\nall workers finished → scoring (section 6)")
        break
    time.sleep(30)

## 6. Score (judge + table)

Asks the judge (Kimi K3) only about label pairs it has not seen before (answers are cached in
`results/traser_bench/judge_cache.json` and pushed with the results), then writes `results/traser_bench/README.md`
with our numbers next to the paper's. It scores **every** dataset that has predictions, so after the last session
the table is complete. Can be run on its own at any time (no GPU needed).

In [ ]:
report = E.evaluate(log=print)
display(Markdown(E.write_report(report)))

## 7. Push the results (do this before the session ends)

In a terminal (or the Source Control panel):
```
cd /kaggle/working/svg2-implementation
git add results/traser_bench
git commit -m "TRASER bench: <dataset> predictions"
git push
```
Only small text files are pushed (predictions, labels, scores); videos and masks stay on Kaggle's temporary disk.